# Libraries Import and Base Path initialization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import boxmot.trackers.ocsort.ocsort as o
print(dir(o))

from boxmot import Boxmot
help(Boxmot)

from boxmot.trackers.ocsort.ocsort import OcSort
help(OcSort)

## Osnet location

Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF

To: /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/models/osnet_x0_25_msmt17.pt

100%|██████████| 3.06M/3.06M [00:00<00:00, 15.4MB/s]
SUCCESS  | Loaded pretrained weights from /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/models/osnet_x0_25_msmt17.pt

In [1]:
import os, cv2, json, math, pickle, random
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO
from boxmot.trackers.ocsort.ocsort import OcSort
# from boxmot.trackers.deepocsort.deepocsort import DeepOcSort
import torch

from mmengine.config import Config
from mmengine.registry import MODELS
from mmengine.runner import load_checkpoint
from mmaction.apis import init_recognizer

# # This function will now work correctly because we are running from the cloned directory
from mmaction.utils import register_all_modules
register_all_modules(init_default_scope=True) # We set the scope manually later

#Colab Base Path
# base_path = "/content/drive/MyDrive/SMT 6/CV/UAS"

#Local Base Path
base_path = ""

# === Dataset Paths ===
# data_path = os.path.join(base_path, "match_videos")
data_path = os.path.join(base_path, "practice_videos")

# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) TWT 2024.mp4")
# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) 1 round.mp4")
# video_path = os.path.join(data_path, "lowhigh(bryan) vs ninjakilla(law)_1round.mp4")
video_path = os.path.join(data_path, "Bryan_LR_Complete.mp4")

# annotation_path = os.path.join(base_path, "match_videos/Knee(Bryan) vs Double(Law) TWT 2024.json")
# annotation_path = os.path.join(data_path, "Knee_reindexed.json")
annotation_path = os.path.join(data_path, "Bryan_LR_Complete.json")

labels_path = os.path.join(data_path, "move_labels.json") # move class labels
filtered_labels_path = os.path.join(data_path, "move_labels_filtered.json") # selected/remapped move labels
skeleton_dataset = os.path.join(data_path, "skeleton_dataset.pkl") # STGCN++ dataset
output_dir = os.path.join(data_path, "frames")
kp_dir = os.path.join(data_path)

# video_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_trimmed.mp4")
# annotation_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_2.json")
# output_dir = os.path.join(base_path, "Bryan_2/frames")

# === Load YOLO detection and pose models ===
# yolo = YOLO("yolo11s.pt")
# yolo = YOLO("yolo11m.pt")
# yolo = YOLO("yolo26s.pt")
yolo  = YOLO("runs/detect/practice_m/weights/best.pt")
# yolo  = YOLO("twt_practice_m.pt")
# yolo_pose = YOLO("yolo11s-pose.pt")
yolo_pose = YOLO("yolo11l-pose.pt")
yolo.to("mps")
yolo_pose.to("mps")

# os.makedirs(output_dir, exist_ok=True)

/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


YOLO(
  (model): PoseModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_runni

# YOLO and ByteTrack character tracking

In [2]:
def make_tracker():
    return OcSort(
        half=True,
        device="mps",
        max_age=90,
        min_hits=2,
        iou_threshold=0.15,
        det_thresh=0.20,
        # iou_threshold=0.25,
        # det_thresh=0.30,
    )

tracker = make_tracker()

def yolo_detect(image):
    result = yolo(image, conf=0.5, iou=0.35, classes=[0])[0]

    # the results of yolo.predict contains list of object per frame it detects, for example image will have 1 object in the list
    # while video will have as many object in it as the video frames
    # we will access the first object as it is an image
    # Play with conf(minimum conf to be detected) and 
    # iou (how much the boxes can overlap to be considered the same object)
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 6), dtype=np.float32)

    xyxy = result.boxes.xyxy.cpu().numpy() # convert boxes x1, y1, x2, y2 of selected object to numpy, then to int
    conf = result.boxes.conf.cpu().numpy().reshape(-1, 1) # convert boxes confidence of selected object to numpy
    cls = result.boxes.cls.cpu().numpy().reshape(-1, 1) # convert boxes class of selected object to numpy

    detections = np.hstack((xyxy, conf, cls))
    # stack boxes, conf, cls horizontally (it only accepts tuple so we encapsulate it with double ()

    # print("Results xyxy", result.boxes.xyxy) # print the x1, y1, x2, y2 from boxes of the object
    # print("detections", detections, type(detections))
    return detections

def ocsort_tracking(detections, image): 
    if detections.shape[0] == 0:
        return None
    
    tracks = tracker.update(detections, image)
    if tracks is None or len(tracks) == 0:
        return None
    
    ids = tracks[:, 4].astype(int).reshape(-1, 1)
    boxes = tracks[:, :4].astype(int)
    id_box_array = np.hstack((ids, boxes))
    return id_box_array

def pad_box(h, w, box, padding=0):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    return np.array([_, x1, y1, x2, y2], dtype=int)
    
def crop_roi(frame, box):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    # print("x1: ", x1, "y1: ", y1, "x2: ", x2, "y2: ", y2)
    return frame[y1:y2, x1:x2]

    # Matrix (Memory) Coordinates: NumPy thinks in (row, column).
    # Because images are processed top-to-bottom, a row corresponds to 
    # the vertical y position, and a column corresponds to the horizontal x position.

def display_roi(player1_roi, player2_roi):
    fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (4, 8))
    axes[0].imshow(player1_roi)
    axes[1].imshow(player2_roi)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def is_timer_missing(frame, edge_threshold=50):
    """
    Checks the Tekken timer UI using Edge Detection.
    Returns True if the sharp metallic borders of the numbers are missing.
    """
    y1, y2 = 26, 86
    x1, x2 = 600, 680
    timer_roi = frame[y1:y2, x1:x2]
    
    gray = cv2.cvtColor(timer_roi, cv2.COLOR_BGR2GRAY)
    
    # cv2.Canny highlights sharp transitions. 
    # The silver border of the font will light up brilliantly here.
    edges = cv2.Canny(gray, 100, 200)
    
    # Count how many 'edge' pixels exist in that small box
    edge_count = cv2.countNonZero(edges)
    
    # If the count drops below the threshold, the timer is gone.
    return edge_count < edge_threshold

def is_cinematic_zoom(boxes, frame_height, threshold=0.75): # Raised to 85%
    """
    Detects if the camera has zoomed in brutally for a Rage Art, Tornado, or K.O.
    """
    if boxes is None or len(boxes) == 0:
        return False
        
    for box in boxes:
        box_h = box[3] - box[1] 
        if box_h > (frame_height * threshold):
            return True
            
    return False

WARNING  Max age > max observations, increasing size of max observations...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001

# YOLO Pose Function

In [3]:
def empty_keypoints():
    return np.full((17, 3), np.nan, dtype=np.float32)

def box_iou_xyxy(box, boxes):
    box = np.asarray(box, dtype=np.float32)
    boxes = np.asarray(boxes, dtype=np.float32)
    if boxes.size == 0:
        return np.array([], dtype=np.float32)

    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)

    box_area = max(0, box[2] - box[0]) * max(0, box[3] - box[1])
    boxes_area = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
    union = box_area + boxes_area - inter
    return np.divide(inter, union, out=np.zeros_like(inter, dtype=np.float32), where=union > 0)

def choose_pose_candidate(boxes_xyxy, box_scores, target_box=None):
    if boxes_xyxy.shape[0] == 0:
        return 0
    if target_box is None:
        return int(np.argmax(box_scores))

    target_box = np.asarray(target_box, dtype=np.float32)
    ious = box_iou_xyxy(target_box, boxes_xyxy)

    target_center = np.array([(target_box[0] + target_box[2]) / 2, (target_box[1] + target_box[3]) / 2], dtype=np.float32)
    candidate_centers = np.column_stack(((boxes_xyxy[:, 0] + boxes_xyxy[:, 2]) / 2, (boxes_xyxy[:, 1] + boxes_xyxy[:, 3]) / 2))
    distances = np.linalg.norm(candidate_centers - target_center, axis=1)
    target_diag = max(1.0, np.linalg.norm([target_box[2] - target_box[0], target_box[3] - target_box[1]]))
    center_score = 1.0 - np.clip(distances / target_diag, 0.0, 1.0)

    match_score = (2.0 * ious) + center_score + (0.25 * box_scores)
    return int(np.argmax(match_score))

def track_box_to_roi_box(track_box, crop_box):
    _, tx1, ty1, tx2, ty2 = track_box
    _, cx1, cy1, _, _ = crop_box
    return np.array([tx1 - cx1, ty1 - cy1, tx2 - cx1, ty2 - cy1], dtype=np.float32)

def run_yolopose(image, input_size=192, conf=0.25, iou=0.35, target_box=None):
    if image is None or image.size == 0:
        return empty_keypoints()

    roi_height, roi_width = image.shape[:2]
    if roi_height == 0 or roi_width == 0:
        return empty_keypoints()

    result = yolo_pose(image, imgsz=input_size, conf=conf, iou=iou, verbose=False)[0]
    if result.keypoints is None or len(result.keypoints) == 0:
        return empty_keypoints()

    xy = result.keypoints.xy.cpu().numpy()
    kp_conf = result.keypoints.conf
    if kp_conf is None:
        scores = np.ones(xy.shape[:2], dtype=np.float32)
    else:
        scores = kp_conf.cpu().numpy()

    if xy.shape[0] == 0:
        return empty_keypoints()

    if result.boxes is not None and len(result.boxes) > 0:
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        box_scores = result.boxes.conf.cpu().numpy()
        best_idx = choose_pose_candidate(boxes_xyxy, box_scores, target_box)
        best_idx = min(best_idx, xy.shape[0] - 1)
    else:
        mean_scores = np.nanmean(scores, axis=1)
        best_idx = 0 if np.all(np.isnan(mean_scores)) else int(np.nanargmax(mean_scores))

    keypoints = empty_keypoints()
    points_xy = xy[best_idx]
    point_scores = scores[best_idx]
    num_points = min(17, points_xy.shape[0])

    keypoints[:num_points, 0] = points_xy[:num_points, 1] / roi_height
    keypoints[:num_points, 1] = points_xy[:num_points, 0] / roi_width
    keypoints[:num_points, 2] = point_scores[:num_points]
    return keypoints

def denormalize_points(points, original_height, original_width, input_size=None):
    """
    Converts normalized YOLO-pose ROI keypoints back to ROI pixel coordinates.
    """
    y, x, c = points
    if np.isnan(y) or np.isnan(x):
        return (np.nan, np.nan, c)

    y_abs = y * original_height
    x_abs = x * original_width
    return (y_abs, x_abs, c)

def normalize_points_to_full_frame(kp_array, box, full_height, full_width):
    """
    Normalize array of keypoints to full frame size
    """
    _, x1, y1, x2, y2 = box
    roi_height = y2-y1
    roi_width = x2-x1

    kp_full = []
    for kp in kp_array:
        y_roi, x_roi, c = denormalize_points(kp, roi_height, roi_width)
        if np.isnan(y_roi) or np.isnan(x_roi):
            kp_full.append([np.nan, np.nan, c])
            continue

        y_full = (y_roi + y1) / full_height
        x_full = (x_roi + x1) / full_width
        kp_full.append([y_full, x_full, c])

    return np.array(kp_full)
        

def draw_keypoints(frame, keypoints, color, full_height, full_width, score_threshold=0.0):
    if keypoints is None:
        return

    for kp in keypoints:
        y, x, c = kp
        if not np.isfinite(y) or not np.isfinite(x):
            continue
        if np.isfinite(c) and c < score_threshold:
            continue

        y_px = int(np.clip(y * full_height, 0, full_height - 1))
        x_px = int(np.clip(x * full_width, 0, full_width - 1))
        cv2.circle(frame, (x_px, y_px), 3, color, thickness=2, lineType=cv2.LINE_AA)


def interpolate_points(player_kp):
    player_kp = np.array(player_kp)
    for kp in range(player_kp.shape[1]):
        for coord in range(player_kp.shape[2]):
            data = player_kp[:, kp, coord]
            nans = np.isnan(data) # nans mask example: [true, false, true] based on the positions
            if np.any(~nans):
                data[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), data[~nans])
            player_kp[:, kp, coord] = data
            print(data)
    return player_kp
# for kp in player1_kp:
#     print(kp)
#     y, x, c = denormalize_points(kp, original_height, original_width)

# YOLO Pose Extraction

In [4]:
input_size = 384
cap = cv2.VideoCapture(video_path)
print(video_path, os.path.exists(video_path))

player1_kp = []
player2_kp = []
player1_track = []  # [id, x1, y1, x2, y2] or NaNs per frame
player2_track = []  # [id, x1, y1, x2, y2] or NaNs per frame
other_track = []    # [id, x1, y1, x2, y2] or NaNs per frame
player_set = False
player1_id, player2_id = None, None
player1_box, player2_box, other_box = None, None, None
player1_kp_full, player2_kp_full, other_kp_full = None, None, None
frame_count = 0

missing_timer_frames = 0
buffer_limit = 10  # 10 frames ignores brief juggles, but catches the 15-frame practice reset
tracking_active = True

cinematic_frames = 0
cinematic_buffer = 5 # Wait 5 frames to confirm a zoom

while cap.isOpened(): # read every single frame of the video
    ret, frame_bgr = cap.read()
    if not ret:
        print("End of video")
        break

    original_height, original_width = frame_bgr.shape[:2]

    # --- 1. CHECK THE TIMER ---
    if is_timer_missing(frame_bgr):
        missing_timer_frames += 1
    else:
        missing_timer_frames = 0
        tracking_active = True # Timer is clearly visible, tracking is safe

    # --- HANDLE THE RESET STATE ---
    if missing_timer_frames > buffer_limit:
        
        # Only print and wipe memory the FIRST time we cross the threshold
        if tracking_active:
            print(f"Scene transition detected at frame {frame_count}. Pausing tracking...")
            tracking_active = False 
            
            tracker = make_tracker()
            # last_p1_box = None
            # last_p2_box = None
            # If using an OC-SORT instance, destroy/re-init it here.
            player_set = False

        # We are in a reset state (black screen or waiting for fade-in).
        # Append empty frames to keep your temporal arrays perfectly aligned for the pipeline.
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        player1_track.append(np.full(5, np.nan, dtype=np.float32))
        player2_track.append(np.full(5, np.nan, dtype=np.float32))
        other_track.append(np.full(5, np.nan, dtype=np.float32))
        
        frame_count += 1
        
        # Skip the rest of the loop entirely. Do not run YOLO.
        continue


    # --- 2. THE CINEMATIC CHECK ---
    # Run your raw YOLO detection ONCE per frame
    detections = yolo_detect(frame_bgr) 
    
    # # Check if the boxes are massive (camera zoomed in)
    # if is_cinematic_zoom(detections, frame_height, threshold=0.85):
    #     cinematic_frames += 1
    # else:
    #     cinematic_frames = 0

    # if cinematic_frames > cinematic_buffer:
    #     print(f"Cinematic zoom detected at frame {frame_count}. Pausing tracking...")
        
    #     # Append NaNs because no actual gameplay is happening
    #     player1_kp.append(np.full((17, 3), np.nan))
    #     player2_kp.append(np.full((17, 3), np.nan))
        
    #     # Wipe the player memory! 
    #     # Characters often land in different spots after a Tornado/Rage Art.
    #     # This forces the logic to re-evaluate who is on the left/right when zooming out.
    #     player_set = False 
        
    #     frame_count += 1
    #     continue # Skip OC-SORT and YOLO pose

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    id_box_array = ocsort_tracking(detections, frame_bgr) # 2d array containing id_box from p1 and 2
    # print(id_box_array)

    if id_box_array is None or id_box_array.size == 0:
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        player1_track.append(np.full(5, np.nan, dtype=np.float32))
        player2_track.append(np.full(5, np.nan, dtype=np.float32))
        other_track.append(np.full(5, np.nan, dtype=np.float32))
        frame_count += 1
        continue

    # if frame_count == 0:
    #     original_height, original_width = frame_bgr.shape[:2]
    #     print("Original height, original_width", original_height, original_width)
    #     player1_id = id_box_array[0, 0]
    #     player2_id = id_box_array[1, 0]

    if id_box_array is not None and id_box_array.shape[0] >= 2 and not player_set:
        centers_x = (id_box_array[:,1] + id_box_array[:,3]) / 2
        order = np.argsort(centers_x)           # left -> right
        player1_id = int(id_box_array[order[0], 0])
        player2_id = int(id_box_array[order[1], 0]) 

        # ABOVE EXTRA STEPS MIGHT NOT BE NECESSARY
        # player1_id = id_box_array[0, 0]
        # player2_id = id_box_array[1, 0]
        player_set = True

    p1_exist = any(id_box_array[:, 0] == player1_id)
    p2_exist = any(id_box_array[:, 0] == player2_id)

    other_mask = ~np.isin(id_box_array[:, 0], [player1_id, player2_id])
    has_other = np.any(other_mask)
    player1_track_frame = np.full(5, np.nan, dtype=np.float32)
    player2_track_frame = np.full(5, np.nan, dtype=np.float32)
    other_track_frame = np.full(5, np.nan, dtype=np.float32)
    
    if p1_exist:
        player1_box = id_box_array[id_box_array[:, 0] == player1_id][0]
        player1_track_frame = player1_box.astype(np.float32)
        # print("p1 box", player1_box)
        player1_box_padded = pad_box(original_height, original_width, player1_box)
        player1_roi = crop_roi(frame_bgr, player1_box_padded)
        player1_target_box = track_box_to_roi_box(player1_box, player1_box_padded)
        player1_kp_raw = run_yolopose(player1_roi, input_size, target_box=player1_target_box)
        player1_kp_full = normalize_points_to_full_frame(player1_kp_raw, player1_box_padded, original_height, original_width)
        player1_kp.append(player1_kp_full)
    else:
        player1_box = None
        player1_box_padded = None
        player1_kp_full = None
        player1_kp.append(np.full((17, 3), np.nan))

    if p2_exist:
        player2_box = id_box_array[id_box_array[:, 0] == player2_id][0]
        player2_track_frame = player2_box.astype(np.float32)
        player2_box_padded = pad_box(original_height, original_width, player2_box)
        player2_roi = crop_roi(frame_bgr, player2_box_padded)
        player2_target_box = track_box_to_roi_box(player2_box, player2_box_padded)
        player2_kp_raw = run_yolopose(player2_roi, input_size, target_box=player2_target_box)
        player2_kp_full = normalize_points_to_full_frame(player2_kp_raw, player2_box_padded, original_height, original_width)
        player2_kp.append(player2_kp_full)
    else:
        player2_box = None
        player2_box_padded = None
        player2_kp_full = None
        player2_kp.append(np.full((17, 3), np.nan))

    if has_other:
        other_box = id_box_array[other_mask][0]
        other_track_frame = other_box.astype(np.float32)
        other_box_padded = pad_box(original_height, original_width, other_box)
        other_roi = crop_roi(frame_bgr, other_box_padded)
        other_target_box = track_box_to_roi_box(other_box, other_box_padded)
        other_kp_raw = run_yolopose(other_roi, input_size, target_box=other_target_box)
        other_kp_full = normalize_points_to_full_frame(other_kp_raw, other_box_padded, original_height, original_width)
    else:
        other_box = None
        other_box_padded = None
        other_kp_full = None


    # Visualization: draw player 1 (green) and player 2 (red)
    color_p1 = (0, 255, 0)  # green (B, G, R)
    color_p2 = (0, 0, 255)  # red
    
    # Boxes + IDs
    if player1_box is not None:
        obj_id, x1, y1, x2, y2 = player1_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p1, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p1, thickness=1, lineType=cv2.LINE_AA)
    
    if player2_box is not None:
        obj_id, x1, y1, x2, y2 = player2_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p2, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    if other_box is not None:
        obj_id, x1, y1, x2, y2 = other_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (255, 0, 0), thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    # Keypoints
    draw_keypoints(frame_bgr, player1_kp_full, color_p1, original_height, original_width)
    draw_keypoints(frame_bgr, player2_kp_full, color_p2, original_height, original_width)

    player1_track.append(player1_track_frame)
    player2_track.append(player2_track_frame)
    other_track.append(other_track_frame)

    cv2.imshow('Process feed', frame_bgr)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    # plt.imshow(frame_rgb)
    # plt.show()

    frame_count += 1

cap.release()

# V10 behavior: full-video interpolation before saving keypoints.
# player1_kp = interpolate_points(player1_kp)
# player2_kp = interpolate_points(player2_kp)
# V12 behavior: keep raw NaNs so missing keypoints stay visible to training.
player1_kp = np.asarray(player1_kp, dtype=np.float32)
player2_kp = np.asarray(player2_kp, dtype=np.float32)
player1_track = np.asarray(player1_track, dtype=np.float32)
player2_track = np.asarray(player2_track, dtype=np.float32)
other_track = np.asarray(other_track, dtype=np.float32)

np.save(os.path.join(kp_dir, "player1_kp"), player1_kp)
np.save(os.path.join(kp_dir, "player2_kp"), player2_kp)
np.save(os.path.join(kp_dir, "player1_track"), player1_track)
np.save(os.path.join(kp_dir, "player2_track"), player2_track)
np.save(os.path.join(kp_dir, "other_track"), other_track)

practice_videos/Bryan_LR_Complete.mp4 True

0: 384x640 2 fighters, 139.2ms
Speed: 2.9ms preprocess, 139.2ms inference, 32.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 26.9ms
Speed: 1.3ms preprocess, 26.9ms inference, 6.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.1ms
Speed: 1.1ms preprocess, 17.1ms inference, 6.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.0ms
Speed: 1.2ms preprocess, 17.0ms inference, 6.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.2ms
Speed: 1.0ms preprocess, 16.2ms inference, 6.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.8ms
Speed: 1.1ms preprocess, 16.8ms inference, 6.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.5ms
Speed: 1.1ms preprocess, 16.5ms inference, 6.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.2ms
Speed: 1.2ms preprocess, 17

WARNING  Max age > max observations, increasing size of max observations...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


0: 384x640 2 fighters, 16.2ms
Speed: 1.2ms preprocess, 16.2ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.6ms
Speed: 1.1ms preprocess, 16.6ms inference, 6.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.6ms
Speed: 1.1ms preprocess, 16.6ms inference, 6.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.7ms
Speed: 1.1ms preprocess, 16.7ms inference, 6.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.6ms
Speed: 1.1ms preprocess, 16.6ms inference, 6.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.0ms
Speed: 1.3ms preprocess, 17.0ms inference, 6.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.0ms
Speed: 1.3ms preprocess, 17.0ms inference, 6.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.0ms
Speed: 1.3ms preprocess, 20.0ms inference, 6.3ms postprocess per image at

WARNING  Max age > max observations, increasing size of max observations...

...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


0: 384x640 2 fighters, 16.7ms
Speed: 1.1ms preprocess, 16.7ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.0ms
Speed: 1.2ms preprocess, 17.0ms inference, 6.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.0ms
Speed: 1.4ms preprocess, 17.0ms inference, 6.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.0ms
Speed: 1.1ms preprocess, 17.0ms inference, 6.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.0ms
Speed: 1.2ms preprocess, 17.0ms inference, 6.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.8ms
Speed: 1.2ms preprocess, 16.8ms inference, 6.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.7ms
Speed: 1.2ms preprocess, 17.7ms inference, 6.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.8ms
Speed: 1.0ms preprocess, 16.8ms inference, 6.3ms postprocess per image at

WARNING  Max age > max observations, increasing size of max observations...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


0: 384x640 2 fighters, 16.9ms
Speed: 1.2ms preprocess, 16.9ms inference, 6.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.8ms
Speed: 1.2ms preprocess, 17.8ms inference, 6.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.3ms
Speed: 1.0ms preprocess, 17.3ms inference, 7.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.2ms
Speed: 0.9ms preprocess, 17.2ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.7ms
Speed: 1.3ms preprocess, 16.7ms inference, 6.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.7ms
Speed: 1.2ms preprocess, 16.7ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 17.5ms
Speed: 1.3ms preprocess, 17.5ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 16.7ms
Speed: 1.2ms preprocess, 16.7ms inference, 6.9ms postprocess per image at

Sure! Here's a concise summary of everything we discussed, formatted in markdown for easy reference:

---

## 📚 Summary: MMAction2 Skeleton Dataset Format & Preparation Steps

Skeleton-based Action Recognition in MMAction2 doesn’t require splitting the original video, but it **does require splitting the keypoint data** into action-based segments.

---

### 🧬 Dataset Format Overview (`.pkl`)

```python
{
  "split": {
    "train": ["clip1", "clip2", ...],
    "val": ["clip7", "clip8", ...],
    ...
  },
  "annotations": [
    {
      "frame_dir": "clip1",
      "label": 0,
      "img_shape": (1080, 1920),
      "original_shape": (1080, 1920),
      "total_frames": 87,
      "keypoint": np.ndarray([M, T, V, C]),
      "keypoint_score": np.ndarray([M, T, V])
    },
    ...
  ]
}
```

- **`frame_dir`**: Unique name for each clip
- **`label`**: Action class (int)
- **`img_shape` & `original_shape`**: Optional frame resolution
- **`total_frames`**: Frames in the segment
- **`keypoint`**: Shape `[M x T x V x C]` (people, frames, joints, coords)
- **`keypoint_score`**: Confidence for each keypoint `[M x T x V]`

---

### ⚙️ Steps to Prepare from a Long Video

If you already extracted full video keypoints:

1. **Use Annotations**  
   Get frame ranges for each action from your annotation file.

2. **Slice Keypoint Arrays**  
   Extract each action clip from the full keypoint array using its frame indices.

3. **Assign Clip Identifiers**  
   Name each segment like `clip001`, `clip002`, etc.

4. **Group into Splits**  
   Organize clip names into `'train'`, `'val'`, etc. inside the `split` dictionary.

5. **Build Annotations List**  
   For each clip, create a dictionary with all required fields and add it to `annotations`.

6. **Save to Pickle**  
   Combine `split` and `annotations` into a Python dict and save as `.pkl`.

---

Want me to build a sample Python script to help automate these steps? Happy to dive in! 💻

# Prepare dataloader

In [ ]:
## Deprecated: 5-Move Dataset Preparation
'''
This cell was used for the early 5-move ST-GCN++ experiment.
It is not used in the final pipeline. The final dataset is generated by
make_all_moves_idle_dataset.py with all annotated moves, idle class,
fixed 30-frame windows, no global keypoint interpolation, and temporal
offset augmentation -3,0,6.
'''
# # JSON Annotation
# with open(annotation_path) as f:
#     annotations = json.load(f)

# with open(labels_path) as l:
#     labels = json.load(l)

# # Choose which moves go into skeleton_dataset.pkl.
# # Use names from move_labels.json or original numeric label IDs. Set to None to include all moves.
# INCLUDED_MOVE_LABELS = [
#     "Bryan 1",
#     "Bryan 4, 1",
#     "Bryan b+1",
#     "Bryan df+2",
#     "Bryan f, b+2",
# ]
# REMAPPED_FILTERED_LABELS = True
# TRAIN_SPLIT_RATIO = 0.8
# RANDOM_SEED = 42

# def build_label_filter(labels, included_move_labels=None, remap=True):
#     reverse_labels = {label_id: move_name for move_name, label_id in labels.items()}

#     if included_move_labels is None:
#         selected_move_names = [name for name, _ in sorted(labels.items(), key=lambda item: item[1])]
#     else:
#         selected_move_names = []
#         for label_ref in included_move_labels:
#             if isinstance(label_ref, int):
#                 if label_ref not in reverse_labels:
#                     raise ValueError(f"Unknown label id: {label_ref}")
#                 move_name = reverse_labels[label_ref]
#             else:
#                 move_name = label_ref
#                 if move_name not in labels:
#                     raise ValueError(f"Unknown move label: {move_name}")

#             if move_name not in selected_move_names:
#                 selected_move_names.append(move_name)

#     label_remap = {}
#     filtered_labels = {}
#     for new_label, move_name in enumerate(selected_move_names):
#         old_label = labels[move_name]
#         label_remap[old_label] = new_label if remap else old_label
#         filtered_labels[move_name] = label_remap[old_label]

#     return set(selected_move_names), label_remap, filtered_labels

# included_move_classes, label_remap, filtered_labels = build_label_filter(
#     labels,
#     INCLUDED_MOVE_LABELS,
#     REMAPPED_FILTERED_LABELS
# )

# with open(filtered_labels_path, "w") as f:
#     json.dump(filtered_labels, f, indent=2)

# # Keypoints
# player1_kp = np.load(os.path.join(kp_dir, "player1_kp.npy"))
# player2_kp = np.load(os.path.join(kp_dir, "player2_kp.npy"))

# # STGCN++ compliant dataset structure
# datasets = {
#     "split": {
#         "train": [],
#         "val": []
#     },
#     "annotations": []
# }

# # sequences_id = list(annotations.keys())[0]
# # data = annotations[sequences_id]

# # print(sequences_id, data)
# # print(annotations.keys())
# # print(player1_kp[0])

# def kp_slicing(kp):
#     conf = kp[:, :, 2].astype(np.float32)
#     x_coord = kp[:, :, 1]
#     y_coord = kp[:, :, 0]
#     coords = np.stack((x_coord, y_coord), axis = -1).astype(np.float32)
#     # print("confidence:", conf.shape)
#     # print("xy:", coords.shape)

#     # Expand dimensions to meet MMAction2 requirements
#     # keypoint expected: [M, T, V, C] -> (1, Frames, 17, 2)
#     keypoint_mm = np.expand_dims(coords, axis=0)
    
#     # keypoint_score expected: [M, T, V] -> (1, Frames, 17)
#     score_mm = np.expand_dims(conf, axis=0).astype(np.float32)
    
#     return keypoint_mm, score_mm

# def train_val_split(annotations, train_ratio=TRAIN_SPLIT_RATIO, seed=RANDOM_SEED):
#     rng = random.Random(seed)
#     annotations_by_label = {}
#     for ann in annotations:
#         annotations_by_label.setdefault(ann["label"], []).append(ann)

#     train_dirs = []
#     val_dirs = []
#     for label, label_annotations in sorted(annotations_by_label.items()):
#         label_annotations = label_annotations.copy()
#         rng.shuffle(label_annotations)

#         if train_ratio >= 1.0 or len(label_annotations) == 1:
#             split_idx = len(label_annotations)
#         else:
#             split_idx = int(round(train_ratio * len(label_annotations)))
#             split_idx = max(1, min(split_idx, len(label_annotations) - 1))

#         train_dirs.extend(ann["frame_dir"] for ann in label_annotations[:split_idx])
#         val_dirs.extend(ann["frame_dir"] for ann in label_annotations[split_idx:])

#     rng.shuffle(train_dirs)
#     rng.shuffle(val_dirs)
#     return train_dirs, val_dirs

# class_counts = {move_name: 0 for move_name in filtered_labels}
# skipped_counts = {}

# for sequence_id, data in annotations.items():
#     # print(sequence_id)
#     # print(data)
#     player = data["player"]
#     start_frame = data["start_frame"]
#     end_frame = data["end_frame"] + 1  # +1 IS TEMPT FIX FOR TWT ANOTATION
#     move_class = data['character'] + " " + data['move']

#     if move_class not in included_move_classes:
#         skipped_counts[move_class] = skipped_counts.get(move_class, 0) + 1
#         continue

#     # print(start_frame, end_frame)

#     # The desired data is [person [frames of the actions [kp of frame [coord of kp]]]]
#     if player == "player1":
#         kp_sequence = player1_kp[start_frame:end_frame]
                                       
#     else:
#         kp_sequence = player2_kp[start_frame:end_frame]

#     # print(kp_sequence.shape)
#     kp, kp_score = kp_slicing(kp_sequence)
#     label = label_remap[labels[move_class]]
#     # print("label:", label)
#     clip_annotation = {
#         "frame_dir": sequence_id,
#         "label": label,
#         "img_shape": (720, 1280),
#         "original_shape": (720, 1280),
#         "total_frames": len(kp_sequence),
#         "keypoint": kp,
#         "keypoint_score": kp_score
#     }
#     datasets["annotations"].append(clip_annotation)
#     class_counts[move_class] += 1

# if not datasets["annotations"]:
#     raise ValueError("No clips were added. Check INCLUDED_MOVE_LABELS against move_labels.json.")

# train, val = train_val_split(datasets["annotations"])

# datasets["split"]['train'].extend(train)
# datasets["split"]['val'].extend(val)

# with open(skeleton_dataset, "wb") as f:
#     pickle.dump(datasets, f)

# print(f"Saved {len(datasets['annotations'])} clips to {skeleton_dataset}")
# print(f"Saved filtered label map to {filtered_labels_path}")
# print("Included classes:")
# for move_name, new_label in sorted(filtered_labels.items(), key=lambda item: item[1]):
#     old_label = labels[move_name]
#     print(f"  {new_label}: {move_name} (old label {old_label}, clips {class_counts[move_name]})")
# print(f"Skipped {sum(skipped_counts.values())} clips from {len(skipped_counts)} other move labels")

# Prepare STGCN++ model

In [6]:
config_file = "https://github.com/open-mmlab/mmaction2/blob/main/configs/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d.py"
checkpoint = "https://download.openmmlab.com/mmaction/v1.0/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d_20221228-19a34aba.pth"

# Loading data back
with open(skeleton_dataset, 'rb') as f:
    restored_data = pickle.load(f)

print(restored_data)


{'split': {'train': ['p2_sequence_116', 'p2_sequence_113', 'p1_sequence_103', 'p2_sequence_21', 'p1_sequence_36', 'p2_sequence_52', 'p2_sequence_20', 'p1_sequence_133', 'p1_sequence_2', 'p1_sequence_37', 'p2_sequence_149', 'p2_sequence_49', 'p1_sequence_130', 'p1_sequence_34', 'p2_sequence_84', 'p1_sequence_39', 'p2_sequence_145', 'p1_sequence_7', 'p2_sequence_148', 'p1_sequence_71', 'p1_sequence_66', 'p1_sequence_68', 'p1_sequence_69', 'p2_sequence_50', 'p2_sequence_119', 'p1_sequence_129', 'p1_sequence_65', 'p1_sequence_5', 'p2_sequence_151', 'p2_sequence_82', 'p1_sequence_135', 'p2_sequence_114', 'p1_sequence_33', 'p2_sequence_81', 'p2_sequence_18', 'p1_sequence_100', 'p1_sequence_4', 'p2_sequence_55', 'p1_sequence_97', 'p2_sequence_117'], 'val': ['p2_sequence_53', 'p2_sequence_23', 'p2_sequence_17', 'p1_sequence_101', 'p1_sequence_98', 'p2_sequence_87', 'p2_sequence_85', 'p2_sequence_146', 'p1_sequence_1', 'p1_sequence_132']}, 'annotations': [{'frame_dir': 'p1_sequence_1', 'label':

# Train STGCN++ all moves

In [ ]:
%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS"

# Build dataset from every annotated move plus a small idle class.
# V13 note: use raw/no-global-interpolation player*_kp.npy from the v12 extraction cell.
# Important: positive move clips are expanded to fixed 30-frame windows
# so training looks closer to real rolling-window inference.
# V13 tests a recovery-biased offset set: less startup than -6,0,6, but not as aggressive as 0,3,6.
# Optional later experiment / v14: change to --positive-offsets=0,3,6 for stronger recovery bias.

!.venv/bin/python make_all_moves_idle_dataset.py \
  --positive-window-size 30 \
  --positive-window-mode center \
  --positive-offsets=-3,0,6 \
  --idle-count 30 \
  --idle-window-size 30 \
  --idle-stride 15 \
  --exclude-margin 15 \
  --min-mean-score 0.35

%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/mmaction2"

!MPLCONFIGDIR=/private/tmp/matplotlib \
PYTORCH_ENABLE_MPS_FALLBACK=1 \
CUDA_VISIBLE_DEVICES=-1 \
../.venv/bin/python tools/train.py \
  configs/skeleton/stgcnpp/tekken_stgcn_all_moves_idle.py \
  --work-dir work_dirs/tekken_stgcn_v13_all_moves_idle_no_interp_offsets_m3_0_6

/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS
Saved dataset: /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/practice_videos/skeleton_dataset_all_moves_idle.pkl
Saved labels:  /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/practice_videos/move_labels_all_moves_idle.json
Annotated clips: 480
Idle candidates: 499
Idle chosen:     30
Train/val sizes: 408/102
Class counts:
   0: Bryan 1 = 30
   1: Bryan 4, 1 = 30
   2: Bryan 3+4 = 30
   3: Bryan b+1 = 30
   4: Bryan df+2 = 30
   5: Bryan uf+4 = 30
   6: Bryan f, b+2 = 30
   7: Bryan b+2, 1 = 30
   8: Bryan db+3 = 30
   9: Bryan df+3 = 30
  10: Bryan SWA.1 = 33
  11: Bryan SWA.3 = 30
  12: Bryan SLS.2, 4 = 30
  13: Bryan SLS.1+2 = 27
  14: Bryan WS.1 = 30
  15: Bryan WS.3 = 30
  16: idle = 30
/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/mmaction2
06/18 10:33:18 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: darwin
    Python: 3.10.11 (v3.10.11:7d4cc5aa85, Apr  4 2023, 19:05:19) [Cla

# Run Inference

In [1]:
%cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS"

!.venv/bin/python tekken_video_inference.py \
  --video practice_videos/Bryan_LR_Complete.mp4 \
  --output practice_videos/tekken_stgcn_demo_v13_m3_0_6.mp4 \
  --config mmaction2/configs/skeleton/stgcnpp/tekken_stgcn_all_moves_idle.py \
  --labels practice_videos/move_labels_all_moves_idle.json \
  --checkpoint mmaction2/work_dirs/tekken_stgcn_v13_all_moves_idle_no_interp_offsets_m3_0_6/best_acc_top1_epoch_*.pth \
  --pose-model yolo11l-pose.pt \
  --pose-imgsz 384 \
  --hide-labels idle \
  --action-conf 0.55 \
  --stable-predictions 2 \
  --kp-interpolation none

/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS


/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Loads checkpoint by local backend from path: mmaction2/work_dirs/tekken_stgcn_v13_all_moves_idle_no_interp_offsets_m3_0_6/best_acc_top1_epoch_16.pth
/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/mmengine/runner/checkpoint.py:347: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  checkpoint = torch.load(filename, map_location=map_location)
WARNING  Max age > max observations, increasing size of max observations...     
INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2,            
         iou_threshold=0.15, per_class=False, asso_func=iou, min_conf=0.1,      
         delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01,             
         Q_s_scaling=0.0001                                                     
Input: practice_videos/Bryan_LR_Complete.mp4
Output: practice_videos/tekken_stgcn_demo_v1